# Aim 2 · B2 문항축약 v7 — 엘라스틱넷 vs L2, 주적재량 vs 공통성(h²) 4-way 비교

**v6 → v7 변경점**

1. **v6의 숨은 버그를 고침**: v6 `fit_cuts()`는 `LogisticRegression(..., l1_ratio=0.5, C=0.1, ...)`을 호출하면서
   `penalty=` 인자를 지정하지 않았습니다. sklearn `LogisticRegression`의 기본 `penalty`는 **`"l2"`**이고,
   `penalty="elasticnet"`이 아니면 `l1_ratio`는 **조용히 무시**됩니다. 즉 v6은 "엘라스틱넷"이라고 로그를 찍었지만
   실제로는 **순수 L2(릿지) 로지스틱**을 돌리고 있었습니다. (팀원의 `sjlee` 스크립트에서 L2가 elastic-net보다
   지표가 더 잘 나온 것도 같은 계열의 현상 — L2는 상관된 21개 ADL 문항을 전부 살려두고 완만히만 축소하기
   때문에, 이번 표본·차원에서는 변수선택 페널티가 있는 elastic-net보다 CV 성능이 유리하게 나올 수 있습니다.)
2. **`fit_cuts()`에 `penalty` 인자를 명시적으로 추가**해서 `"elasticnet"`(l1_ratio=0.5, 팀 사전지정)과 `"l2"`를
   같은 코드 경로·같은 solver(`saga`)·같은 C로 정확히 A/B 비교할 수 있게 했습니다.
3. **문항축소 곡선을 4개 조합**으로 전부 생성합니다: {주적재량, 공통성h²} × {elastic-net, L2}.
   기존 §11~§14(주적재량 vs 공통성)의 취지는 유지하면서, 페널티 종류까지 축을 하나 더 추가한 것입니다.
4. 진단용 셀(§4)에서 **v6 버그를 그대로 재현 → 수정 버전과 나란히 비교**해서, "penalty 인자를 빼먹으면
   실제로 L2가 나온다"는 것을 이 노트북 안에서 직접 확인합니다.

나머지 데이터 준비·요인분석 로직은 v6과 동일합니다 (자체완결형, `adl_wide.csv`/`baseline_sample.csv` 기반).


In [ ]:
# =================================================================
# 설정 — Colab 구글드라이브
# =================================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/2026 urp/preprocessed")
OUT_DIR = Path("/content/drive/MyDrive/2026 urp/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DM_PATH = DATA_DIR / "dm_filtered.csv"
DS_PATH = DATA_DIR / "ds_wide.csv"
ADL_PATH = DATA_DIR / "adl_wide.csv"
MMSE_PATH = DATA_DIR / "mmse_wide.csv"
SUP_PATH = DATA_DIR / "supervision_time.csv"
BASELINE_PATH = DATA_DIR / "baseline_sample.csv"

ALPHA = 0.05


In [ ]:
# 요인분석 / 한글 폰트 패키지 (Colab 기본 설치 X)
!pip install factor_analyzer koreanize-matplotlib -q

%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib  # 그래프 한글 깨짐 방지

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo

TRIALS = ["AD-1061", "AD-1063", "AD-1064"]
PENALTIES = ["elasticnet", "l2"]   # 이번 v7에서 비교할 두 페널티
C_DEFAULT = 0.1                    # sjlee Gn 스크립트에서 중첩CV로 선택된 대표 C

BADL = ["ADL0101", "ADL0102", "ADL0103", "ADL0104", "ADL0105", "ADL0106B"]
IADL = [
    "ADL0106A", "ADL0107A", "ADL0110A", "ADL0111A", "ADL0112A", "ADL0113A",
    "ADL0114A", "ADL0115A", "ADL0116A", "ADL0116B", "ADL0117A", "Q18",
    "ADL0121A", "ADL0122Q", "ADL0123L",
]
ITEMS = BADL + IADL  # 21문항 (원 QSTESTCD 코드; Q18은 아래에서 조립함)
NI = len(ITEMS)

LAB = {"ADL0101":"Q1먹기","ADL0102":"Q2걷기","ADL0103":"Q3화장실","ADL0104":"Q4목욕","ADL0105":"Q5몸단장",
 "ADL0106B":"Q6b옷입기","ADL0106A":"Q6a옷고르기","ADL0107A":"Q7전화","ADL0110A":"Q10설거지","ADL0111A":"Q11식사준비",
 "ADL0112A":"Q12집안일","ADL0113A":"Q13빨래","ADL0114A":"Q14가전","ADL0115A":"Q15외출","ADL0116A":"Q16a쇼핑",
 "ADL0116B":"Q16b지불","ADL0117A":"Q17금전","Q18":"Q18혼자있기","ADL0121A":"Q21글쓰기","ADL0122Q":"Q22취미","ADL0123L":"Q23가전사용"}

_lines = []
def log(s=""):
    print(s)
    _lines.append(s)


## 1. 표본 구축 — baseline_sample + adl_wide 병합

v6과 동일: `baseline_sample.csv`에서 `in_aim1_2_sample == True`인 사람만 남기고, 같은 기저 방문(`VISITNUM==2.0`)의
`adl_wide.csv`(원 문항 응답 + `A0_harmonized` + `A1_2015_stage`)를 USUBJID로 조인합니다.

In [ ]:
df_base = pd.read_csv(BASELINE_PATH)
df_adl = pd.read_csv(ADL_PATH)

df_base_use = df_base[df_base["in_aim1_2_sample"] == True].copy()
df_adl_base = df_adl[df_adl["VISITNUM"] == 2.0].copy()

m = df_base_use.merge(
    df_adl_base,
    on=["STUDYID", "USUBJID"],
    how="inner",
    suffixes=("", "_adl"),
)

# Q18은 단일 컬럼이 아니라 ADL0118A/B/C 세 하위문항으로 쪼개져 있음 -> __resolved 세 값의 평균을 대표값으로 사용
# (v6과 동일한 임시 처리. 정식 채점표 확정되면 교체 필요)
Q18_SUBITEMS = ["ADL0118A__resolved", "ADL0118B__resolved", "ADL0118C__resolved"]
missing_sub = [c for c in Q18_SUBITEMS if c not in m.columns]
if missing_sub:
    print("⚠️ Q18 하위문항 중 없는 것:", missing_sub)
else:
    m["Q18"] = m[Q18_SUBITEMS].mean(axis=1)

# 21개 문항 + ds_stage + A0_harmonized + A1_2015_stage가 전부 있는지 확인
required_cols = ITEMS + ["ds_stage", "A0_harmonized", "A1_2015_stage"]
missing_cols = [c for c in required_cols if c not in m.columns]
if missing_cols:
    print("⚠️ 다음 컬럼이 없습니다:", missing_cols)
    ITEMS = [c for c in ITEMS if c in m.columns]
    BADL = [c for c in BADL if c in ITEMS]
    IADL = [c for c in IADL if c in ITEMS]
    NI = len(ITEMS)
    print("→ 남은 문항수:", NI, ITEMS)

m = m.dropna(subset=["ds_stage"]).reset_index(drop=True)
before_n = len(m)
m = m.dropna(subset=["A0_harmonized"]).reset_index(drop=True)
if len(m) < before_n:
    print(f"⚠️ A0_harmonized 결측 {before_n - len(m)}건 제외 → 남은 표본 {len(m)}명")

log(f"# Aim 2 · B2 문항축약 v7 — 엘라스틱넷 vs L2 × 주적재량 vs 공통성(h2)\n")
log(f"공통표본 n={len(m)} (STUDYID별: {m['STUDYID'].value_counts().to_dict()})")
m.head()


## 2. 지표 함수 — mae / kappa / 중증놓침 / 비대칭분류

v6과 동일 (`miss()`/`asym()`은 재구성한 정의라는 점도 동일).

In [ ]:
def mae(t, pred):
    t = np.asarray(t, dtype=float); pred = np.asarray(pred, dtype=float)
    return float(np.mean(np.abs(t - pred)))

def kap(t, pred):
    t = np.asarray(t, dtype=float); pred = np.asarray(pred, dtype=float)
    mask = ~(np.isnan(t) | np.isnan(pred))
    t = t[mask].astype(int); pred = pred[mask].astype(int)
    if len(t) == 0:
        return float("nan")
    return float(cohen_kappa_score(t, pred, weights="quadratic"))

def miss(t, pred):
    """중증놓침: 실제 4~5단계(중증)인데 예측이 4 미만으로 나온 비율."""
    t = np.asarray(t, dtype=float); pred = np.asarray(pred, dtype=float)
    mask = ~(np.isnan(t) | np.isnan(pred))
    t = t[mask]; pred = pred[mask]
    severe = t >= 4
    if severe.sum() == 0:
        return 0.0
    return float(np.mean(pred[severe] < 4))

def asym(P, t4, t5):
    """비대칭 임계값 분류: 중증(4~5단계)을 덜 놓치는 쪽으로 편향."""
    pred = np.argmax(P, axis=1).astype(float)
    p45 = P[:, 4] + P[:, 5]
    pred = np.where(p45 >= t4, 4.0, pred)
    pred = np.where(P[:, 5] >= t5, 5.0, pred)
    return pred

def metrics_all(t, pred):
    t = np.asarray(t, dtype=float); pred = np.asarray(pred, dtype=float)
    diff = pred - t
    return {
        "MAE": mae(t, pred),
        "중증놓침(%)": miss(t, pred) * 100,
        "kappa": kap(t, pred),
        "과분류(%)": (diff > 0).mean() * 100,
        "과소분류(%)": (diff < 0).mean() * 100,
        "정확일치(%)": (diff == 0).mean() * 100,
        "인접일치±1(%)": (np.abs(diff) <= 1).mean() * 100,
        "RMSE": float(np.sqrt(np.mean(diff ** 2))),
    }


## 3. CV 유틸 함수 — 부분비례오즈(PO) 로지스틱, leave-one-trial-out

**여기가 v6과 가장 크게 달라진 부분입니다.** `fit_cuts()`에 `penalty` 인자를 명시적으로 받아서
`"elasticnet"`(l1_ratio=0.5)과 `"l2"`를 정확히 지정합니다. v6은 이 인자가 없어서 `l1_ratio`가 무시되고
항상 L2로 적합되고 있었습니다.

In [ ]:
def fit_cuts(Ztr, y, penalty="elasticnet", C=C_DEFAULT):
    """절단별(0/1..5) 로지스틱 5개. penalty="elasticnet"이면 l1_ratio=0.5(사전지정),
    penalty="l2"이면 순수 릿지. solver는 둘 다 saga로 고정 — 페널티 종류만 바뀌도록 통제."""
    kwargs = dict(solver="saga", C=C, max_iter=3000, tol=1e-3, random_state=0)
    if penalty == "elasticnet":
        kwargs.update(penalty="elasticnet", l1_ratio=0.5)
    elif penalty == "l2":
        kwargs.update(penalty="l2")
    else:
        raise ValueError(f"알 수 없는 penalty: {penalty}")
    return {k: LogisticRegression(**kwargs).fit(Ztr, (y >= k).astype(int)) for k in range(1, 6)}

def probs(models, Z):
    g = {k: models[k].predict_proba(Z)[:, 1] for k in range(1, 6)}
    P = np.zeros((Z.shape[0], 6)); P[:, 0] = 1 - g[1]
    for k in range(1, 5): P[:, k] = g[k] - g[k + 1]
    P[:, 5] = g[5]; P = np.clip(P, 1e-9, None); P /= P.sum(1, keepdims=True); return P

def tune_tau(Ptr, ds_tr, a0m):
    grid = np.round(np.arange(0.05, 0.55, 0.02), 3); best = None
    for t4 in grid:
        for t5 in grid:
            if t5 < t4: continue
            p = asym(Ptr, t4, t5)
            if miss(ds_tr, p) <= a0m + 1e-9:
                key = mae(ds_tr, p)
                if best is None or key < best[0]: best = (key, t4, t5)
    if best is None:
        c = [(miss(ds_tr, asym(Ptr, a, b)), mae(ds_tr, asym(Ptr, a, b)), a, b) for a in grid for b in grid if b >= a]
        _, _, t4, t5 = min(c); best = (0, t4, t5)
    return best[1], best[2]

def prep(m):
    folds = {}
    for ho in TRIALS:
        tri = np.array(m.index[m.STUDYID != ho]); tei = np.array(m.index[m.STUDYID == ho])
        imp = SimpleImputer(strategy="median").fit(m.loc[tri, ITEMS])
        Xtr = imp.transform(m.loc[tri, ITEMS]); Xte = imp.transform(m.loc[tei, ITEMS])
        y = m.loc[tri, "ds_stage"].astype(int).values; ds_tr = m.loc[tri, "ds_stage"].values
        a0m = miss(ds_tr, m.loc[tri, "A0_harmonized"].values)
        folds[ho] = dict(tei=tei, Xtr=Xtr, Xte=Xte, y=y, ds_tr=ds_tr, a0m=a0m)
    return folds

def eval_subset(m, folds, keep, penalty="elasticnet", C=C_DEFAULT):
    t = m["ds_stage"].values; pred = np.zeros(len(m))
    for ho in TRIALS:
        f = folds[ho]
        sc = StandardScaler().fit(f["Xtr"][:, keep])
        mdl = fit_cuts(sc.transform(f["Xtr"][:, keep]), f["y"], penalty=penalty, C=C)
        Ptr = probs(mdl, sc.transform(f["Xtr"][:, keep])); Pte = probs(mdl, sc.transform(f["Xte"][:, keep]))
        t4, t5 = tune_tau(Ptr, f["ds_tr"], f["a0m"])
        pred[f["tei"]] = asym(Pte, t4, t5)
    return metrics_all(t, pred)


## 4. 진단 — v6 버그 재현 vs 수정본 (21문항 전체, 단일 fold)

`penalty=` 없이 `l1_ratio=0.5, C=0.1`만 넘기면 sklearn이 조용히 `penalty="l2"`로 적합한다는 것을
21문항 전체·AD-1061 held-out 한 fold에서 직접 재현해서 확인합니다. "버그 버전"과 "penalty='l2' 명시 버전"의
계수가 완전히 동일하면 v6이 실제로 L2를 돌리고 있었다는 뜻입니다.

In [ ]:
ho_demo = "AD-1061"
tri_demo = m.index[m.STUDYID != ho_demo]
imp_demo = SimpleImputer(strategy="median").fit(m.loc[tri_demo, ITEMS])
sc_demo = StandardScaler().fit(imp_demo.transform(m.loc[tri_demo, ITEMS]))
Z_demo = sc_demo.transform(imp_demo.transform(m.loc[tri_demo, ITEMS]))
y_demo = (m.loc[tri_demo, "ds_stage"].astype(int).values >= 3).astype(int)

# v6 버그 그대로 재현: penalty 인자 없이 l1_ratio만 넘김 -> 내부적으로 penalty="l2"
buggy = LogisticRegression(solver="saga", l1_ratio=0.5, C=C_DEFAULT,
                            max_iter=3000, tol=1e-3, random_state=0).fit(Z_demo, y_demo)
# 명시적 L2
explicit_l2 = LogisticRegression(solver="saga", penalty="l2", C=C_DEFAULT,
                                  max_iter=3000, tol=1e-3, random_state=0).fit(Z_demo, y_demo)
# 진짜 엘라스틱넷(수정본)
explicit_en = LogisticRegression(solver="saga", penalty="elasticnet", l1_ratio=0.5, C=C_DEFAULT,
                                  max_iter=3000, tol=1e-3, random_state=0).fit(Z_demo, y_demo)

max_diff_buggy_vs_l2 = np.max(np.abs(buggy.coef_ - explicit_l2.coef_))
max_diff_buggy_vs_en = np.max(np.abs(buggy.coef_ - explicit_en.coef_))
n_zero_en = int((np.abs(explicit_en.coef_) < 1e-8).sum())
n_zero_l2 = int((np.abs(explicit_l2.coef_) < 1e-8).sum())

log("\n## v6 penalty 버그 재현 (21문항, held-out AD-1061, 절단 Y>=3)")
log(f"- '버그' 계수 vs 명시적 L2 계수 최대차이: {max_diff_buggy_vs_l2:.10f}  (0이면 완전 동일 = v6이 사실상 L2였음)")
log(f"- '버그' 계수 vs 명시적 elastic-net 계수 최대차이: {max_diff_buggy_vs_en:.6f}")
log(f"- L2 계수 중 정확히 0인 개수: {n_zero_l2} / {NI}  (릿지는 절대 0을 만들지 않음)")
log(f"- Elastic-net 계수 중 정확히 0인 개수: {n_zero_en} / {NI}  (라쏘 성분이 일부를 0으로 침)\n")


## 5. 기준선(A0/A1) 성능 및 폴드 준비

In [ ]:
a1_mae = mae(m["ds_stage"].values, m["A1_2015_stage"].values)
a1_metrics = metrics_all(m["ds_stage"].values, m["A1_2015_stage"].values)
a0_metrics = metrics_all(m["ds_stage"].values, m["A0_harmonized"].values)
a0_miss = a0_metrics["중증놓침(%)"]

log(f"A1(2015판) 기준: MAE={a1_metrics['MAE']:.3f}, 중증놓침={a1_metrics['중증놓침(%)']:.1f}%, kappa={a1_metrics['kappa']:.3f}")
log(f"A0(2025판 조화) 기준: MAE={a0_metrics['MAE']:.3f}, 중증놓침={a0_metrics['중증놓침(%)']:.1f}%, kappa={a0_metrics['kappa']:.3f}")

folds = prep(m)

pd.DataFrame([
    {"모델": "A1(2015판)", **a1_metrics},
    {"모델": "A0(2025판 조화)", **a0_metrics},
]).round(3)


## 6. 요인분석 — 적합성 확인 및 축 개수 결정(Kaiser)

In [ ]:
imp_full = SimpleImputer(strategy="median").fit(m[ITEMS])
X_full = imp_full.transform(m[ITEMS])
sc_full = StandardScaler().fit(X_full)
Z_full = sc_full.transform(X_full)

chi2, p_bart = calculate_bartlett_sphericity(Z_full)
kmo_all, kmo_model = calculate_kmo(Z_full)
log(f"Bartlett 구형성 검정: chi2={chi2:.1f}, p={p_bart:.4g} (p<{ALPHA} 이면 FA 적합)")
log(f"KMO 표본적합도: {kmo_model:.3f} (0.6 이상 권장)")

fa_ev = FactorAnalyzer(rotation=None)
fa_ev.fit(Z_full)
eigvals, _ = fa_ev.get_eigenvalues()
n_factors = max(2, int((eigvals > 1).sum()))

log(f"고유값(상위 8개): {np.round(eigvals[:8], 3).tolist()}")
log(f"Kaiser 기준(고유값>1) 축 개수: {n_factors}")

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(eigvals) + 1), eigvals, "o-")
plt.axhline(1, color="red", linestyle="--", linewidth=1, label="고유값=1")
plt.xlabel("요인 번호"); plt.ylabel("고유값"); plt.title("Scree plot")
plt.legend(); plt.tight_layout()
plt.savefig(OUT_DIR / "b2v7_fa_scree.png", dpi=150)
plt.show()


## 7. 회전 요인적재량 및 주요인 배정

`n_factors`가 마음에 안 들면 다음 셀의 `n_factors`를 직접 지정하세요(예: `n_factors = 2`).

In [ ]:
# n_factors = 2  # 필요하면 이 줄 주석을 풀고 직접 지정하세요.

fa = FactorAnalyzer(n_factors=n_factors, rotation="varimax")
fa.fit(Z_full)
loadings = fa.loadings_

primary = np.argmax(np.abs(loadings), axis=1)

load_df = pd.DataFrame(
    loadings, index=[LAB[it] for it in ITEMS],
    columns=[f"요인{i+1}" for i in range(n_factors)]
)
load_df["주요인"] = [f"요인{p+1}" for p in primary]
load_df["영역"] = ["BADL" if it in BADL else "IADL" for it in ITEMS]
load_df["주적재량"] = [loadings[i, primary[i]] for i in range(NI)]
load_df = load_df.sort_values(["주요인", "주적재량"])

log("\n## 요인적재량 (varimax 회전)")
log(load_df.round(3).to_string())
load_df.round(3).to_csv(OUT_DIR / "b2v7_fa_loadings.csv", encoding="utf-8-sig")

load_df.round(3)


In [ ]:
plt.figure(figsize=(6, 8))
plt.imshow(loadings, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(label="적재량")
plt.yticks(range(NI), [LAB[it] for it in ITEMS], fontsize=8)
plt.xticks(range(n_factors), [f"요인{i+1}" for i in range(n_factors)])
plt.title("요인적재량 히트맵")
plt.tight_layout()
plt.savefig(OUT_DIR / "b2v7_fa_loadings_heatmap.png", dpi=150)
plt.show()


## 8. 제거 순서 두 가지 생성 — 주적재량 vs 공통성(h²)

지금까지는 v6과 동일합니다. 여기서부터는 **두 제거 순서를 먼저 둘 다 만들어두고**, 아래 §9에서
{순서 2종} × {페널티 2종} = **4개 조합**을 한 번에 순회하도록 구조를 바꿨습니다.

$$h_i^2 = \sum_{j=1}^{n\_factors} \lambda_{ij}^2$$

In [ ]:
def build_removal_order_by_value(metric_values, primary, n_factors):
    """요인별로 값이 작은(약한) 문항부터 라운드로빈으로 제거 순서를 만듦.
    metric_values로 '주적재량'(부호 있는 값) 또는 'h2'(공통성) 둘 다 넣을 수 있음."""
    queues = {f: [] for f in range(n_factors)}
    for i in range(len(primary)):
        queues[primary[i]].append((metric_values[i], i))
    for f in queues:
        queues[f].sort(key=lambda x: x[0])

    order = []
    f = 0
    total = sum(len(q) for q in queues.values())
    while len(order) < total:
        if queues[f]:
            _, idx = queues[f].pop(0)
            order.append(idx)
        f = (f + 1) % n_factors
    return order

# (1) 주적재량 기준 — v6 §7과 동일한 순서(부호 있는 적재량 자체로 정렬)
fa_order_primary = build_removal_order_by_value(
    [loadings[i, primary[i]] for i in range(NI)], primary, n_factors)

# (2) 공통성(h2) 기준 — v6 §12과 동일
h2 = (loadings ** 2).sum(axis=1)
fa_order_h2 = build_removal_order_by_value(h2, primary, n_factors)

h2_df = pd.DataFrame({
    "코드": ITEMS, "라벨": [LAB[it] for it in ITEMS],
    "주요인": [f"요인{primary[i]+1}" for i in range(NI)],
    "주적재량": loadings[np.arange(NI), primary].round(3),
    "공통성(h2)": h2.round(3),
})
h2_df["주적재량_순위"] = h2_df["주적재량"].rank(method="first").astype(int)
h2_df["공통성_순위"] = h2_df["공통성(h2)"].rank(method="first").astype(int)
h2_df["순위차이"] = h2_df["공통성_순위"] - h2_df["주적재량_순위"]
h2_df = h2_df.sort_values("공통성(h2)").reset_index(drop=True)

log("\n## 공통성(h2) vs 주적재량 순위 비교")
log(h2_df.to_string(index=False))
h2_df.to_csv(OUT_DIR / "b2v7_communality_vs_primary_loading.csv", index=False, encoding="utf-8-sig")

log("\n## 제거 순서 (주적재량 기준, 앞이 먼저 제거)")
for rank, idx in enumerate(fa_order_primary, start=1):
    log(f"{rank:2d}. {LAB[ITEMS[idx]]}  (요인{primary[idx]+1}, 적재={loadings[idx, primary[idx]]:.3f})")

log("\n## 제거 순서 (공통성 h2 기준, 앞이 먼저 제거)")
for rank, idx in enumerate(fa_order_h2, start=1):
    log(f"{rank:2d}. {LAB[ITEMS[idx]]}  (요인{primary[idx]+1}, h2={h2[idx]:.3f})")

diff_ranks = [r + 1 for r in range(NI) if fa_order_primary[r] != fa_order_h2[r]]
log(f"\n두 순서가 처음 갈라지는 순번: {diff_ranks[0] if diff_ranks else '없음(완전히 동일)'}")

h2_df


## 9. 문항수별 성능 곡선 — 4개 조합 {주적재량, 공통성h²} × {elastic-net, L2}

`run_curve()` 하나로 4개 조합을 모두 순회합니다. 각 조합은 21문항→4문항까지 1문항씩 줄여가며
leave-one-trial-out CV(3개 fold)로 재적합합니다. 21문항 기준 4조합 × 18단계 × 3fold = 216회 적합이라
Colab에서 몇 분 걸릴 수 있습니다.

In [ ]:
def run_curve(order, penalty, label, C=C_DEFAULT):
    rows = []
    for step in range(0, NI - 3):  # step=0..NI-4 -> 문항수 k=NI..4
        k = NI - step
        removed_idx = order[:step]
        keep = sorted(set(range(NI)) - set(removed_idx))

        if step > 0:
            j = order[step - 1]
            removed_code = ITEMS[j]
            removed_label = LAB[removed_code]
            removed_factor = f"요인{primary[j] + 1}"
        else:
            removed_code, removed_label, removed_factor = "-", "-", "-"

        met = eval_subset(m, folds, keep, penalty=penalty, C=C)
        champ = "✅" if (met["MAE"] < a1_mae and met["중증놓침(%)"] <= a0_miss + 1.0) else "❌"

        rows.append({
            "방법": label, "패널티": penalty, "문항수": k,
            "직전제거_코드": removed_code, "직전제거_라벨": removed_label, "제거축": removed_factor,
            **met,
            "A1대비MAE(앞섬)": a1_mae - met["MAE"], "둘다이김": champ,
            "누적제거문항목록": ", ".join(LAB[ITEMS[i]] for i in removed_idx) if removed_idx else "-",
            "남은문항목록": ", ".join(LAB[ITEMS[i]] for i in keep),
        })
    return pd.DataFrame(rows)


COMBOS = [
    ("주적재량", fa_order_primary, "elasticnet"),
    ("주적재량", fa_order_primary, "l2"),
    ("공통성h2", fa_order_h2, "elasticnet"),
    ("공통성h2", fa_order_h2, "l2"),
]

curves = {}
for label, order, pen in COMBOS:
    key = f"{label}_{pen}"
    log(f"\n### 진행: {label} × {pen}")
    curves[key] = run_curve(order, pen, label=f"{label}/{pen}")
    log(f"  완료 — 21문항 MAE={curves[key].iloc[0]['MAE']:.3f}, "
        f"4문항 MAE={curves[key].iloc[-1]['MAE']:.3f}")

log("\n## 4조합 문항축소 곡선 생성 완료\n")


## 10. 각 조합 저장

In [ ]:
metric_cols = ["방법", "패널티", "문항수", "직전제거_코드", "직전제거_라벨", "제거축",
               "MAE", "중증놓침(%)", "kappa", "과분류(%)", "과소분류(%)",
               "정확일치(%)", "인접일치±1(%)", "RMSE", "A1대비MAE(앞섬)", "둘다이김"]

for key, df_c in curves.items():
    fname = OUT_DIR / f"b2v7_curve_{key}.csv"
    df_c[metric_cols].round(3).to_csv(fname, index=False, encoding="utf-8-sig")
    log(f">>> 저장: {fname}")

display(curves["주적재량_elasticnet"][metric_cols])


## 11. 4조합 비교 — 같은 문항수에서 어느 조합이 가장 좋은가

패널티(elastic-net vs L2)와 축소 순서(주적재량 vs h²)를 동시에 축으로 놓고, 문항수마다
MAE·중증놓침·가중카파를 나란히 비교합니다.

In [ ]:
compare = curves["주적재량_elasticnet"][["문항수", "MAE", "중증놓침(%)", "kappa"]].rename(
    columns={"MAE": "MAE_주적재량_EN", "중증놓침(%)": "미스_주적재량_EN", "kappa": "kappa_주적재량_EN"})

rename_map = {
    "주적재량_l2":    ("MAE_주적재량_L2",  "미스_주적재량_L2",  "kappa_주적재량_L2"),
    "공통성h2_elasticnet": ("MAE_h2_EN",   "미스_h2_EN",   "kappa_h2_EN"),
    "공통성h2_l2":    ("MAE_h2_L2",   "미스_h2_L2",   "kappa_h2_L2"),
}
for key, (mae_c, miss_c, kap_c) in rename_map.items():
    part = curves[key][["문항수", "MAE", "중증놓침(%)", "kappa"]].rename(
        columns={"MAE": mae_c, "중증놓침(%)": miss_c, "kappa": kap_c})
    compare = compare.merge(part, on="문항수")

compare = compare.sort_values("문항수", ascending=False).reset_index(drop=True)

# 문항수별 MAE 최저 조합 표시
mae_cols = [c for c in compare.columns if c.startswith("MAE_")]
compare["최저MAE_조합"] = compare[mae_cols].idxmin(axis=1).str.replace("MAE_", "", regex=False)

log("\n## 4조합 비교표 (문항수별)")
log(compare.round(3).to_string(index=False))
compare.round(3).to_csv(OUT_DIR / "b2v7_compare_4combos.csv", index=False, encoding="utf-8-sig")

compare.round(3)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
styles = {
    "주적재량_elasticnet": dict(marker="o", linestyle="-",  color="tab:blue",   label="주적재량 · Elastic-net"),
    "주적재량_l2":         dict(marker="o", linestyle="--", color="tab:blue",   label="주적재량 · L2"),
    "공통성h2_elasticnet": dict(marker="s", linestyle="-",  color="tab:orange", label="공통성h2 · Elastic-net"),
    "공통성h2_l2":         dict(marker="s", linestyle="--", color="tab:orange", label="공통성h2 · L2"),
}

ax = axes[0]
for key, df_c in curves.items():
    ax.plot(df_c["문항수"], df_c["MAE"], **styles[key])
ax.axhline(a1_mae, color="gray", linestyle=":", label="A1 기준")
ax.invert_xaxis(); ax.set_xlabel("문항수"); ax.set_ylabel("MAE"); ax.set_title("MAE 비교"); ax.legend(fontsize=8)

ax = axes[1]
for key, df_c in curves.items():
    ax.plot(df_c["문항수"], df_c["중증놓침(%)"], **styles[key])
ax.axhline(a0_miss, color="gray", linestyle=":", label="A0 기준")
ax.invert_xaxis(); ax.set_xlabel("문항수"); ax.set_ylabel("중증놓침(%)"); ax.set_title("중증놓침 비교"); ax.legend(fontsize=8)

ax = axes[2]
for key, df_c in curves.items():
    ax.plot(df_c["문항수"], df_c["kappa"], **styles[key])
ax.invert_xaxis(); ax.set_xlabel("문항수"); ax.set_ylabel("가중카파"); ax.set_title("가중카파 비교"); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / "b2v7_compare_4combos.png", dpi=150)
plt.show()


## 12. 문항수 예산별 "둘 다 이김"(A1보다 MAE 낮고, A0+1%p 이내 중증놓침) 요약

각 조합이 몇 문항까지 줄여도 A1·A0를 동시에 능가하는지, 4조합을 나란히 정리합니다.

In [ ]:
summary_rows = []
for key, df_c in curves.items():
    ok = df_c[df_c["둘다이김"] == "✅"]
    min_items = int(ok["문항수"].min()) if len(ok) else None
    summary_rows.append({
        "조합": key,
        "21문항 MAE": round(df_c.iloc[0]["MAE"], 3),
        "21문항 중증놓침(%)": round(df_c.iloc[0]["중증놓침(%)"], 1),
        "둘다이김 유지 최소 문항수": min_items,
        "최소문항수에서 MAE": round(df_c[df_c["문항수"] == min_items]["MAE"].iloc[0], 3) if min_items else None,
    })

summary_df = pd.DataFrame(summary_rows)
log("\n## 조합별 요약 — 문항 축소 여력")
log(summary_df.to_string(index=False))
summary_df.to_csv(OUT_DIR / "b2v7_summary_4combos.csv", index=False, encoding="utf-8-sig")

REPORT_PATH = OUT_DIR / "b2v7_item_reduction_report.md"
REPORT_PATH.write_text("\n".join(_lines), encoding="utf-8")
print(f">>> 저장: {REPORT_PATH}")

summary_df


## 13. 해석 메모

- §4의 진단 결과, `penalty=` 없이 `l1_ratio`만 넘겼던 v6의 "버그" 계수가 명시적 `penalty="l2"` 계수와
  (수치 오차 범위 내에서) 동일했다면, v6에서 이미 "엘라스틱넷"이 아니라 L2를 돌리고 있었다는 뜻입니다.
  이 노트북(v7)의 elastic-net 결과가 처음으로 "진짜" 엘라스틱넷 성능입니다.
- §11~§12 비교표에서 `최저MAE_조합`이 특정 문항수 구간에서 계속 L2 쪽으로 나온다면, 지난 대화에서 논의한
  것처럼 "21개 문항 대부분이 각자 정보를 가지고 있고 상관이 크게 문제되지 않는 상황(VIF 대부분 <2.5)이라
  변수선택 페널티(L1 성분)의 이점이 크지 않다"는 해석과 일치하는지 확인해보세요.
- 반대로 **문항 수를 줄이는 것 자체가 목표**라면, L2는 절대 계수를 정확히 0으로 만들지 않으므로
  이 곡선에서 L2 조합은 "몇 번째 문항부터 제거해도 성능이 버티는가"만 보여줄 뿐, elastic-net처럼
  "모형이 스스로 필요없다고 판단한 문항"을 알려주지는 않습니다. 최종 축소 문항 리스트를 정할 때는
  elastic-net 조합(어느 순서든) 결과를 우선 참고하는 것이 방법론적으로 더 일관됩니다.
- (여기에 문항수별 실제 해석을 추가하세요 — 특히 §12에서 "둘다이김 유지 최소 문항수"가 4조합 간에
  얼마나 갈리는지, §8에서 두 제거순서가 갈라지는 지점(주로 Q12집안일 ↔ Q22취미 교체) 근방에서
  elastic-net과 L2가 서로 다른 반응을 보이는지.)
- ⚠️ `miss()`/`asym()`/Q18 조립 방식은 v6과 동일하게 재구성/임시처리된 정의입니다.
